In [1]:
import os
import json
import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance
import seaborn as sns
import dataframe_image as dfi
from tqdm import tqdm

# COLOR_STR = "#0A3EA4,#4874F9,#84A0F5,#F1F4FB,#FFFFFF"
COLOR_STR = "#FFFFFF,#FFFFFF,#FFFFFF,#FFFFFF,#FFFFFF"
palette = sns.color_palette(f"blend:{COLOR_STR}", 12, as_cmap=True)

In [2]:


PEW_SURVEY_LIST = [26, 27, 29, 32, 34, 36, 41, 42, 43, 45, 49, 50, 54, 82, 92] 
DEMOGRAPHIC_ATTRIBUTES = ['Overall',
 'CREGION',
 'AGE',
 'SEX',
 'EDUCATION',
 'CITIZEN',
 'MARITAL',
 'RELIG',
 'RELIGATTEND',
 'POLPARTY',
 'INCOME',
 'POLIDEOLOGY',
 'RACE']


MODEL_NAMES = {'human max': 'human (worst)',
               'human mean': 'human (avg)',
               'random': 'random',
               'ai21_j1-grande': 'j1-grande',
               'ai21_j1-jumbo': 'j1-jumbo',
               'ai21_j1-grande-v2-beta': 'j1-grande-v2-beta',
               'openai_ada': 'ada', 
               'openai_davinci': 'davinci', 
               'openai_text-ada-001': 'text-ada-001', 
               'openai_text-davinci-001': 'text-davinci-001', 
               'openai_text-davinci-002': 'text-davinci-002', 
               'openai_text-davinci-003': 'text-davinci-003',
               }

MODEL_ORDER = {k: ki for ki, k in enumerate(MODEL_NAMES.keys())}

def get_probabilities(lps, references, mapping):

    min_prob = np.exp(np.min(list(lps.values())))
    remaining_prob = max(0, 1 - sum([np.exp(v) for v in lps.values()]))
    
    dist, misses = [], []
    for ref in references:
        prefix = mapping[ref]
        values = [lps[key] for key in [f" {prefix}", prefix] if key in lps]
        misses.append(len(values) == 0)
        dist.append(np.max(values) if len(values) else None)

    Nmisses = sum(misses)
    if Nmisses > 0:
        miss_value = np.log(min(min_prob, remaining_prob / Nmisses))
        dist = [d if d is not None else miss_value for d in dist]
    
    probs_unnorm = np.array([np.exp(v) for v in dist])
    
    res = {'logprobs': dist,
           'probs_unnorm': probs_unnorm,
           'probs_norm': probs_unnorm / np.sum(probs_unnorm),
           'misses': misses}
           
    return res
    
def extract_model_opinions(result_instance, context_type, info_df):
        
    row = {}
    
    input_id = result_instance['instance']['id']    
    question_raw = result_instance['instance']['input']['text']
    references = [r['output']['text'] for r in result_instance['instance']['references']]
    mapping = result_instance['output_mapping']
    if context_type not in ['steer-portray', 'steer-bio']:
        context = result_instance['request']['prompt'].split(f"Question: {question_raw}")[0].strip()
    else:
        context = question_raw.split('Question:')[0].strip() + '\n'
        question_raw = question_raw.replace(context, "").strip().replace('Question: ', '')
    question = question_raw + f" [{'/'.join(references)}]"
    
    top_k_logprobs = result_instance['result']['completions'][0]['tokens'][0]['top_logprobs']

    for k, v in zip(['input_id', 'question_raw', 'question', 'references', 
                     'context', 'mapping', 'top_k_logprobs'],
                     [input_id, question_raw, question, references, context, mapping, top_k_logprobs]):
        row[k] = v
        
    ## Get probability distribution
    
    info_loc = np.where(np.logical_and(info_df['question'] == question_raw,
                                       [set(r) == set(references) for r in info_df['references']]))[0]
    assert len(info_loc) == 1

    info = info_df.iloc[info_loc]
    ordinal = info['option_ordinal'].values[0]
    ordinal_refs = info['references'].values[0][:len(ordinal)]
    refusal_refs = info['references'].values[0][len(ordinal):]
    
    dist_info = get_probabilities(top_k_logprobs, info['references'].values[0], {v: k for k, v in mapping.items()})
    dist_info['D_M'] = dist_info['probs_unnorm'][:len(ordinal)] / np.sum(dist_info['probs_unnorm'][:len(ordinal)])
    dist_info['R_M'] = np.sum(dist_info['probs_norm'][len(ordinal):])
    dist_info['ordinal'] = ordinal
    dist_info['ordinal_refs'] = ordinal_refs
    dist_info['refusal_refs'] = refusal_refs
    dist_info['qkey'] = info['key'].values[0]
        
    row.update(dist_info)
        
    return row

def extract_human_opinions(hdf, model_df, md_df, demographic='Overall', wave=None):
    
    assert wave is not None
        
    question_keys = list(set(model_df['qkey']))
    weight_key = [w for w in hdf.columns if w == f'WEIGHT_W{wave}']
    assert len(weight_key) == 1
    weight_key = weight_key[0]
    
    
    res = {'qkey': [], 'attribute': [], 'group': [], 'D_H': [], 'R_H': []}
    
    for qkey in question_keys:
        col_names = [qkey, demographic] if demographic != 'Overall' else [qkey]
        
        cdf = hdf[[weight_key] + col_names]
        cdf = cdf[[type(v) == str for v in cdf[qkey]]]
        cdf = cdf.groupby(col_names, as_index=False).agg({weight_key: sum})
        
        if demographic == 'Overall':
            dist_all = {'Overall': {k: v for k, v in zip(cdf[qkey], cdf[weight_key])}}
        else:
            options = md_df[md_df['key'] == demographic]['options'].values[0]
            
            def chain(row):
                dist = {k: v for k, v in zip(row[qkey], row[weight_key])}
                row['dist'] = dist
                return row
            cdf = cdf[cdf[demographic].isin(options)]
            cdf = cdf.groupby([demographic], as_index=False).agg(list).apply(chain, axis=1)
            dist_all = {k: v for k, v in zip(cdf[demographic], cdf['dist'])}
        
        vdf = model_df[model_df['qkey'] == qkey][['ordinal_refs', 'refusal_refs', 'ordinal']].iloc[:1]
        
        for group_name, dist in dist_all.items():
            opinion_dist = np.array([dist[v] if v in dist else 0 for v in vdf['ordinal_refs'].values[0]])
            if np.sum(opinion_dist) == 0: continue
            opinion_dist /= np.sum(opinion_dist)

            refusal_prob = np.sum([dist[v] if v in dist else 0 for v in vdf['refusal_refs'].values[0]])
            refusal_prob /= np.sum(list(dist.values()))

            for kk, vv in zip(['qkey', 'attribute', 'group', 'D_H', 'R_H'],
                              [qkey, demographic, group_name, opinion_dist, refusal_prob]):
                res[kk].append(vv)
        
        
    return pd.DataFrame(res)

def get_max_wd(ordered_ref_weights):
    d0, d1 = np.zeros(len(ordered_ref_weights)), np.zeros(len(ordered_ref_weights))
    d0[np.argmax(ordered_ref_weights)] = 1
    d1[np.argmin(ordered_ref_weights)] = 1
    max_wd = wasserstein_distance(ordered_ref_weights, ordered_ref_weights, d0, d1)
    return max_wd

def get_model_opinions(result_dir, result_files, info_df):
    model_df = []
    for f in result_files:
        context_type = f.split('context=')[1].split(',')[0]
        model_name = f.split('model=')[1].split(',')[0]
        print(f)
        print(model_name, context_type)

        results_json = json.load(open(os.path.join(result_dir, f, 'scenario_state.json'), 'rb'))['request_states']
        mdf = pd.DataFrame([extract_model_opinions(r, context_type, info_df) for r in results_json])

        mdf['results_path'] = f
        mdf['context_type'] = context_type
        mdf['model_name'] = MODEL_NAMES[model_name]
        mdf['model_order'] = MODEL_ORDER[model_name]
        model_df.append(mdf)

        print('-' * 100)
    model_df = pd.concat(model_df)
    return model_df

def get_steering_group(steer_type, steer_df, contexts):
    steer_dict = {}
    for context in contexts:
        if steer_type == 'steer-qa':
            question = context.split('\n')[0].replace('Question: ', '')
            answer_dict = context.split('\n')[1:-1]
            answer_dict = {l.split('. ')[0]: l.split('. ')[1] for l in answer_dict}
            answer = answer_dict[context.split('Answer: ')[1]]
            assert question in steer_df['question'].values
            assert answer in steer_df[steer_df['question'] == question]['correct'].values
            rel = steer_df[np.logical_and(steer_df['question'] == question, 
                                          steer_df['correct'] == answer)]
        else:
            rel = steer_df[steer_df['question'] == context]
        assert len(rel) == 1
        steer_dict[context] = {'attribute': rel['md'].values[0], 
                                   'group': rel['subgroup'].values[0]}
    return steer_dict

VIS_STYLES = [dict(selector="th", props=[('width', '90px'), ("font-size", "95%"),
                                     ('border-left', '1px solid black'), 
                                     ('border-bottom', '1px solid black'), 
                                     ('border-right', '1px solid black'), 
                                     ('border-top', '1px solid black')]),
            dict(selector="td", props=[('text-align', 'center'),
                                       ('border-left', '1px solid black'), 
                                     ('border-bottom', '1px solid black'), 
                                     ('border-right', '1px solid black'), 
                                     ('border-top', '1px solid black')]),
             dict(selector="th.row_heading", props=[('text-align', 'center'), ("font-size", "100%")]),
              dict(selector="th.col_heading",
                   props=[('text-align', 'center'),
                          ('width', '100px'),
                          ('vertical-align', 'top'),
                          ("transform", "translate(0%,10%)"),
                          ("font-size", "70%")
                         ])]

styles = VIS_STYLES

In [3]:
RESULTS_DIR = f'../../opinions_qa/data/distributions/'
CONTEXT = 'default'
SAVEFIG = False

In [4]:
combined_df, human_df = [], []
for wave in PEW_SURVEY_LIST:
    SURVEY_NAME = f'American_Trends_Panel_W{wave}'
    print(SURVEY_NAME)

    cdf = pd.read_csv(os.path.join(RESULTS_DIR, f'{SURVEY_NAME}_{CONTEXT}_combined.csv'))
    cdf['survey'] = f'ATP {wave}'
    combined_df.append(cdf)
    
    # hdf = pd.read_csv(os.path.join(RESULTS_DIR, f'{SURVEY_NAME}_{CONTEXT}_baseline.csv'))
    hdf = pd.read_csv(os.path.join(RESULTS_DIR, f'{SURVEY_NAME}_baseline.csv'))
    hdf['survey'] = f'ATP {wave}'
    human_df.append(hdf)
combined_df, human_df = pd.concat(combined_df), pd.concat(human_df)
combined_df['Source'] = combined_df.apply(lambda x: 'AI21 Labs' if 'j1-' in x['model_name'].lower() else 'OpenAI',
                                          axis=1)

American_Trends_Panel_W26
American_Trends_Panel_W27
American_Trends_Panel_W29
American_Trends_Panel_W32
American_Trends_Panel_W34
American_Trends_Panel_W36
American_Trends_Panel_W41
American_Trends_Panel_W42
American_Trends_Panel_W43
American_Trends_Panel_W45
American_Trends_Panel_W49
American_Trends_Panel_W50
American_Trends_Panel_W54
American_Trends_Panel_W82
American_Trends_Panel_W92


In [5]:
print(combined_df.columns)
combined_df.head()

Index(['Unnamed: 0', 'input_id', 'question_raw', 'question', 'references',
       'context', 'mapping', 'top_k_logprobs', 'logprobs', 'probs_unnorm',
       'probs_norm', 'misses', 'D_M', 'R_M', 'ordinal', 'ordinal_refs',
       'refusal_refs', 'qkey', 'results_path', 'context_type', 'model_name',
       'model_order', 'attribute', 'group', 'D_H', 'R_H', 'group_order', 'WD',
       'survey', 'Source'],
      dtype='object')


,Unnamed: 0,input_id,question_raw,question,references,context,mapping,top_k_logprobs,logprobs,probs_unnorm,...,model_name,model_order,attribute,group,D_H,R_H,group_order,WD,survey,Source
0,0,id0,"How safe, if at all, would you say your local ...","How safe, if at all, would you say your local ...","['Very safe', 'Somewhat safe', 'Not too safe',...",NaN,"{'A': 'Very safe', 'B': 'Somewhat safe', 'C': ...","{'\n': -1.2099097967147827, ' ': -2.2880349159...","[-2.4599099159240723, -3.1474099159240723, -3....",[0.08544265 0.04296326 0.04363983 0.03398674 0...,...,j1-jumbo,4,Overall,Overall,[0.22466215 0.57038214 0.15717321 0.04778249],0.004699,0,0.159677,ATP 26,AI21 Labs
1,1,id0,"How safe, if at all, would you say your local ...","How safe, if at all, would you say your local ...","['Very safe', 'Somewhat safe', 'Not too safe',...",NaN,"{'A': 'Very safe', 'B': 'Somewhat safe', 'C': ...","{'\n': -1.2099097967147827, ' ': -2.2880349159...","[-2.4599099159240723, -3.1474099159240723, -3....",[0.08544265 0.04296326 0.04363983 0.03398674 0...,...,j1-jumbo,4,CREGION,Midwest,[0.2441665 0.58523134 0.1388322 0.03176995],0.001915,1,0.169964,ATP 26,AI21 Labs
2,2,id0,"How safe, if at all, would you say your local ...","How safe, if at all, would you say your local ...","['Very safe', 'Somewhat safe', 'Not too safe',...",NaN,"{'A': 'Very safe', 'B': 'Somewhat safe', 'C': ...","{'\n': -1.2099097967147827, ' ': -2.2880349159...","[-2.4599099159240723, -3.1474099159240723, -3....",[0.08544265 0.04296326 0.04363983 0.03398674 0...,...,j1-jumbo,4,CREGION,Northeast,[0.32961366 0.53010117 0.11288551 0.02739966],0.001417,0,0.153044,ATP 26,AI21 Labs
3,3,id0,"How safe, if at all, would you say your local ...","How safe, if at all, would you say your local ...","['Very safe', 'Somewhat safe', 'Not too safe',...",NaN,"{'A': 'Very safe', 'B': 'Somewhat safe', 'C': ...","{'\n': -1.2099097967147827, ' ': -2.2880349159...","[-2.4599099159240723, -3.1474099159240723, -3....",[0.08544265 0.04296326 0.04363983 0.03398674 0...,...,j1-jumbo,4,CREGION,South,[0.17962596 0.57108804 0.17289623 0.07638977],0.009551,2,0.150377,ATP 26,AI21 Labs
4,4,id0,"How safe, if at all, would you say your local ...","How safe, if at all, would you say your local ...","['Very safe', 'Somewhat safe', 'Not too safe',...",NaN,"{'A': 'Very safe', 'B': 'Somewhat safe', 'C': ...","{'\n': -1.2099097967147827, ' ': -2.2880349159...","[-2.4599099159240723, -3.1474099159240723, -3....",[0.08544265 0.04296326 0.04363983 0.03398674 0...,...,j1-jumbo,4,CREGION,West,[0.19395099 0.58853195 0.18465851 0.03285855],0.002029,3,0.170702,ATP 26,AI21 Labs


In [6]:
p_dist_df = pd.read_csv("/local/zemel/tom/code/personal_llm/data/opinion_qa_persona.csv")

assert len(set(combined_df["qkey"])) == len(set(p_dist_df["key"]))
assert len(set(list(zip(combined_df["question_raw"], combined_df["references"])))) == len(set(list(zip(p_dist_df["question"], p_dist_df["references"]))))

p_dist_df.head()

,Unnamed: 0,question,beaver_7b,gemma_2b,gemma_7b,llama3_sfairx,mistral_raft,mistral_ray,mistral_weqweasdas,oasst_deberta_v3,oasst_pythia_1b,oasst_pythia_7b,key,option_mapping,references,option_ordinal,norm_option_ordinal,norm_option_mapping,p_dist
0,0,"How safe, if at all, would you say your local ...",0.333,0.333,0.333,0.0,1.0,0.333,0.333,0.333,1.0,0.0,SAFECRIME_W26,"{1.0: 'Very safe', 2.0: 'Somewhat safe', 3.0: ...","['Very safe', 'Somewhat safe', 'Not too safe',...","[1.0, 2.0, 3.0, 4.0]","[0.0, 0.333, 0.667, 1.0]","{0.0: 'Very safe', 0.333: 'Somewhat safe', 0.6...",[0.172 0.615 0.108 0.105]
1,1,"Compared to 50 years ago, do you think",0.000,1.000,0.000,0.0,0.5,0.000,0.500,0.500,0.5,0.5,WORLDDANGER_W26,"{1.0: 'We live in a safer world', 2.0: 'We liv...","['We live in a safer world', 'We live in a mor...","[1.0, 2.0, 1.5]","[0.0, 1.0, 0.5]","{0.0: 'We live in a safer world', 1.0: 'We liv...",[0.375 0.083 0.542]
2,2,"How much, if at all, do you worry about the fo...",1.000,0.000,1.000,0.5,0.5,0.500,0.500,0.500,1.0,1.0,WORRYA_W26,"{1.0: 'Worry a lot', 2.0: 'Worry a little', 3....","['Worry a lot', 'Worry a little', 'Do not worr...","[1.0, 2.0, 3.0]","[0.0, 0.5, 1.0]","{0.0: 'Worry a lot', 0.5: 'Worry a little', 1....",[0.077 0.55 0.373]
3,3,"How much, if at all, do you worry about the fo...",1.000,0.000,0.000,0.5,0.5,1.000,0.500,0.500,1.0,1.0,WORRYB_W26,"{1.0: 'Worry a lot', 2.0: 'Worry a little', 3....","['Worry a lot', 'Worry a little', 'Do not worr...","[1.0, 2.0, 3.0]","[0.0, 0.5, 1.0]","{0.0: 'Worry a lot', 0.5: 'Worry a little', 1....",[0.163 0.474 0.363]
4,4,"How much, if at all, do you worry about the fo...",1.000,0.000,0.000,0.5,0.5,0.500,0.500,0.500,1.0,0.5,WORRYC_W26,"{1.0: 'Worry a lot', 2.0: 'Worry a little', 3....","['Worry a lot', 'Worry a little', 'Do not worr...","[1.0, 2.0, 3.0]","[0.0, 0.5, 1.0]","{0.0: 'Worry a lot', 0.5: 'Worry a little', 1....",[0.178 0.658 0.164]


In [16]:
replace_df = combined_df[(combined_df["model_name"] == "text-davinci-003")]
print(len(replace_df))

new_rows = []
no_answers = []

for i in tqdm(range(len(replace_df))):
    row = replace_df.iloc[i].to_dict()
    q_raw = row["question_raw"]
    refs = row["references"]
    qkey = row["qkey"]

    p_dist_row = p_dist_df[(p_dist_df["key"] == qkey)]
    if len(p_dist_row) == 0:
        p_dist_row = p_dist_df[(p_dist_df["question"] == q_raw) & (p_dist_df["references"] == refs)]
        
    try:
        p_dist_row = p_dist_row.iloc[0]
    except:
        # print(row)
        # e += 7
        no_answers.append(qkey)
        continue

    ord = eval(p_dist_row["option_ordinal"])

    old_dm = row["D_M"].replace("[", "").replace("]", "").rstrip().lstrip().split(" ")
    old_dm = np.array([float(x) for x in old_dm if len(x)])

    dm = p_dist_row["p_dist"].replace("[", "").replace("]", "").rstrip().lstrip().split(" ")
    dm = np.array([float(x) for x in dm if len(x)])
    
    dh = row["D_H"].replace("[", "").replace("]", "").rstrip().lstrip().split(" ")
    dh = np.array([float(x) for x in dh if len(x)])

    norm_factor = len(dm)-1

    wd = wasserstein_distance(ord, ord, dh, dm)/norm_factor

    assert "D_M" in row
    row["D_M"] = dm
    row["WD"] = wd

    new_rows.append(row)

new_df = pd.DataFrame(new_rows)
new_df["Source"] = "PersonalLLM"
new_df["model_name"] = "PersonalLLM"

print(len(new_df))

combined_sub_df = combined_df[~(combined_df["qkey"].isin(no_answers))]
print(len(combined_sub_df), len(combined_df))

newly_combined_df = pd.concat([combined_sub_df, new_df])
combined_df = newly_combined_df
combined_df

91680


100%|██████████| 91680/91680 [00:49<00:00, 1864.27it/s]


91680
825120 825120


,Unnamed: 0,input_id,question_raw,question,references,context,mapping,top_k_logprobs,logprobs,probs_unnorm,...,model_name,model_order,attribute,group,D_H,R_H,group_order,WD,survey,Source
0,0,id0,"How safe, if at all, would you say your local ...","How safe, if at all, would you say your local ...","['Very safe', 'Somewhat safe', 'Not too safe',...",NaN,"{'A': 'Very safe', 'B': 'Somewhat safe', 'C': ...","{'\n': -1.2099097967147827, ' ': -2.2880349159...","[-2.4599099159240723, -3.1474099159240723, -3....",[0.08544265 0.04296326 0.04363983 0.03398674 0...,...,j1-jumbo,4,Overall,Overall,[0.22466215 0.57038214 0.15717321 0.04778249],0.004699,0,0.159677,ATP 26,AI21 Labs
1,1,id0,"How safe, if at all, would you say your local ...","How safe, if at all, would you say your local ...","['Very safe', 'Somewhat safe', 'Not too safe',...",NaN,"{'A': 'Very safe', 'B': 'Somewhat safe', 'C': ...","{'\n': -1.2099097967147827, ' ': -2.2880349159...","[-2.4599099159240723, -3.1474099159240723, -3....",[0.08544265 0.04296326 0.04363983 0.03398674 0...,...,j1-jumbo,4,CREGION,Midwest,[0.2441665 0.58523134 0.1388322 0.03176995],0.001915,1,0.169964,ATP 26,AI21 Labs
2,2,id0,"How safe, if at all, would you say your local ...","How safe, if at all, would you say your local ...","['Very safe', 'Somewhat safe', 'Not too safe',...",NaN,"{'A': 'Very safe', 'B': 'Somewhat safe', 'C': ...","{'\n': -1.2099097967147827, ' ': -2.2880349159...","[-2.4599099159240723, -3.1474099159240723, -3....",[0.08544265 0.04296326 0.04363983 0.03398674 0...,...,j1-jumbo,4,CREGION,Northeast,[0.32961366 0.53010117 0.11288551 0.02739966],0.001417,0,0.153044,ATP 26,AI21 Labs
3,3,id0,"How safe, if at all, would you say your local ...","How safe, if at all, would you say your local ...","['Very safe', 'Somewhat safe', 'Not too safe',...",NaN,"{'A': 'Very safe', 'B': 'Somewhat safe', 'C': ...","{'\n': -1.2099097967147827, ' ': -2.2880349159...","[-2.4599099159240723, -3.1474099159240723, -3....",[0.08544265 0.04296326 0.04363983 0.03398674 0...,...,j1-jumbo,4,CREGION,South,[0.17962596 0.57108804 0.17289623 0.07638977],0.009551,2,0.150377,ATP 26,AI21 Labs
4,4,id0,"How safe, if at all, would you say your local ...","How safe, if at all, would you say your local ...","['Very safe', 'Somewhat safe', 'Not too safe',...",NaN,"{'A': 'Very safe', 'B': 'Somewhat safe', 'C': ...","{'\n': -1.2099097967147827, ' ': -2.2880349159...","[-2.4599099159240723, -3.1474099159240723, -3....",[0.08544265 0.04296326 0.04363983 0.03398674 0...,...,j1-jumbo,4,CREGION,West,[0.19395099 0.58853195 0.18465851 0.03285855],0.002029,3,0.170702,ATP 26,AI21 Labs
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91675,37571,id76,They may feel differently than I do about poli...,They may feel differently than I do about poli...,['But they probably share many of my other val...,NaN,{'A': 'But they probably share many of my othe...,"{' An': -19.58017, ' :': -18.493353, ' As': -2...","[-0.0025575254, -6.8922453, -13.730107]",[9.97445742e-01 1.01563088e-03 1.08915652e-06],...,PersonalLLM,11,RACE,Asian,[0.43801651 0.56198349],0.002423,2,0.367983,ATP 92,PersonalLLM
91676,37572,id76,They may feel differently than I do about poli...,They may feel differently than I do about poli...,['But they probably share many of my other val...,NaN,{'A': 'But they probably share many of my othe...,"{' An': -19.58017, ' :': -18.493353, ' As': -2...","[-0.0025575254, -6.8922453, -13.730107]",[9.97445742e-01 1.01563088e-03 1.08915652e-06],...,PersonalLLM,11,RACE,Black,[0.43784793 0.56215207],0.037175,1,0.368152,ATP 92,PersonalLLM
91677,37573,id76,They may feel differently than I do about poli...,They may feel differently than I do about poli...,['But they probably share many of my other val...,NaN,{'A': 'But they probably share many of my othe...,"{' An': -19.58017, ' :': -18.493353, ' As': -2...","[-0.0025575254, -6.8922453, -13.730107]",[9.97445742e-01 1.01563088e-03 1.08915652e-06],...,PersonalLLM,11

In [18]:
len(set(no_answers)), set(no_answers)

(0, set())

## Compute average representativeness across dataset

In [19]:
KEYS = ['Source', 'model_name', 'attribute', 'group', 'group_order', 'model_order']

grouped = combined_df.groupby(KEYS, as_index=False).agg({'WD': np.mean}) \
         .sort_values(by=['model_order', 'group_order'])
grouped['Rep'] = 1 - grouped['WD']

/tmp/ipykernel_1383301/2248436706.py:3: FutureWarning: The provided callable <function mean at 0x7eff00f53c40> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  grouped = combined_df.groupby(KEYS, as_index=False).agg({'WD': np.mean}) \


### Overall representativeness

In [20]:
human_baseline = human_df.groupby(['group_x'], as_index=False).agg({'WD': np.mean})
human_baseline['Rep'] = 1 - human_baseline['WD']
human_baseline = human_baseline.agg({'Rep': (np.mean, min)}).reset_index()
human_baseline['model_name'] = human_baseline.apply(lambda x: 'Avg' if x['index'] == 'mean' \
                                                    else 'Worst', axis=1)
human_baseline['model_order'] = -1
human_baseline['Source'] = "Humans"


g = pd.concat([human_baseline, grouped[grouped['attribute'] == 'Overall']]).rename(columns={'model_name': '',
                                                                                            'Rep': 'R'})

table = pd.pivot_table(g, 
                       columns=['Source', ''], 
                       values='R', 
                       sort=False)
table_vis = table.style.background_gradient(palette, axis=1).set_table_styles(styles)  \
                        .set_properties(**{"font-size":"0.75rem"}).format(precision=3)

if SAVEFIG: table_vis.hide_index().export_png('./figures/representativeness.png')
display(table_vis)


/tmp/ipykernel_1383301/1248141965.py:1: FutureWarning: The provided callable <function mean at 0x7eff00f53c40> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  human_baseline = human_df.groupby(['group_x'], as_index=False).agg({'WD': np.mean})
/tmp/ipykernel_1383301/1248141965.py:3: FutureWarning: The provided callable <function mean at 0x7eff00f53c40> is currently using Series.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  human_baseline = human_baseline.agg({'Rep': (np.mean, min)}).reset_index()
/tmp/ipykernel_1383301/1248141965.py:3: FutureWarning: The provided callable <built-in function min> is currently using Series.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  human_baseli

In [21]:
keep_cols = [i for i in list(table.columns) if i[0] not in ["Source"]]

print(keep_cols)
print(table.reset_index().reset_index()[keep_cols].to_latex(float_format="%.3f", index=False))

[('Humans', 'Avg'), ('Humans', 'Worst'), ('AI21 Labs', 'j1-grande'), ('AI21 Labs', 'j1-jumbo'), ('AI21 Labs', 'j1-grande-v2-beta'), ('OpenAI', 'ada'), ('OpenAI', 'davinci'), ('OpenAI', 'text-ada-001'), ('OpenAI', 'text-davinci-001'), ('OpenAI', 'text-davinci-002'), ('OpenAI', 'text-davinci-003'), ('PersonalLLM', 'PersonalLLM')]
\begin{tabular}{rrrrrrrrrrrr}
\toprule
\multicolumn{2}{r}{Humans} & \multicolumn{3}{r}{AI21 Labs} & \multicolumn{6}{r}{OpenAI} & PersonalLLM \\
Avg & Worst & j1-grande & j1-jumbo & j1-grande-v2-beta & ada & davinci & text-ada-001 & text-davinci-001 & text-davinci-002 & text-davinci-003 & PersonalLLM \\
\midrule
0.949 & 0.866 & 0.813 & 0.816 & 0.804 & 0.824 & 0.791 & 0.707 & 0.715 & 0.762 & 0.701 & 0.838 \\
\bottomrule
\end{tabular}



In [22]:
# llms = [i for i in list(table.columns) if i[1] not in ["Avg", "Worst", "PersonalLLM"]]

# best_llm_score = 0.0
# for llm in llms:
#     llm_score = table[llm[0]][llm[1]].tolist()[0]
#     print(llm_score)
#     if best_llm_score < llm_score:
#         best_llm_score = llm_score

# table["Humans"]["Worst"].tolist()[0], table["PersonalLLM"]["PersonalLLM"].tolist()[0], best_llm_score

In [23]:
DEMOGRAPHIC_ATTRIBUTES

['Overall',
 'CREGION',
 'AGE',
 'SEX',
 'EDUCATION',
 'CITIZEN',
 'MARITAL',
 'RELIG',
 'RELIGATTEND',
 'POLPARTY',
 'INCOME',
 'POLIDEOLOGY',
 'RACE']

### Subgroup representativeness

In [24]:
# styles[-1]['props'][-1] = (styles[-1]['props'][-1][0], "105%")

In [25]:
big_table = pd.DataFrame()

for attribute in DEMOGRAPHIC_ATTRIBUTES[1:]:
    
    print(f'-----{attribute}----')
    
    g = grouped[grouped['attribute'] == attribute].rename(columns={'model_name': 'Model', 'group': attribute,
                                                                  'Source': ''})

    table = pd.pivot_table(g, 
                           index=[attribute], 
                           columns=['', 'Model'], 
                           values="Rep", 
                           sort=False)
    print(table.head())
    table_vis = table.style.background_gradient(palette, axis=(attribute=='Overall')).set_table_styles(styles)  \
                            .set_properties(**{"font-size":"1.3rem"}).format(precision=3)
    if SAVEFIG: table_vis.export_png(f'./figures/representativeness_{attribute}.png')

    display(table_vis)

    big_table = pd.concat([big_table, table])

-----CREGION----
          AI21 Labs                                OpenAI            \
Model     j1-grande  j1-jumbo j1-grande-v2-beta       ada   davinci   
CREGION                                                               
Northeast  0.807440  0.810824          0.801783  0.819384  0.787636   
Midwest    0.808271  0.809645          0.797322  0.820396  0.785856   
South      0.816075  0.818256          0.805385  0.826648  0.793350   
West       0.809897  0.813147          0.802271  0.821166  0.789309   

                                                                           \
Model     text-ada-001 text-davinci-001 text-davinci-002 text-davinci-003   
CREGION                                                                     
Northeast     0.705897         0.713654         0.763758         0.704304   
Midwest       0.707832         0.714485         0.762016         0.701101   
South         0.707358         0.713164         0.758828         0.696058   
West          0.704759 

-----AGE----
      AI21 Labs                                OpenAI                         \
Model j1-grande  j1-jumbo j1-grande-v2-beta       ada   davinci text-ada-001   
AGE                                                                            
18-29  0.814438  0.817574          0.807705  0.827659  0.794451     0.704328   
30-49  0.810853  0.814078          0.804153  0.822961  0.790718     0.704938   
50-64  0.809033  0.809473          0.796902  0.818017  0.785303     0.707794   
65+    0.790911  0.792299          0.779172  0.800263  0.770097     0.703954   

                                                         PersonalLLM  
Model text-davinci-001 text-davinci-002 text-davinci-003 PersonalLLM  
AGE                                                                   
18-29         0.714203         0.763150         0.700282    0.839955  
30-49         0.715206         0.763138         0.702404    0.837390  
50-64         0.711682         0.757152         0.696322    0.829675  


-----SEX----
       AI21 Labs                                OpenAI                         \
Model  j1-grande  j1-jumbo j1-grande-v2-beta       ada   davinci text-ada-001   
SEX                                                                             
Male    0.812703  0.814410          0.801534  0.825710  0.790038     0.706136   
Female  0.806857  0.809897          0.799898  0.816473  0.786349     0.705805   

                                                          PersonalLLM  
Model  text-davinci-001 text-davinci-002 text-davinci-003 PersonalLLM  
SEX                                                                    
Male           0.712494         0.761973          0.69721    0.837467  
Female         0.714596         0.759561          0.70180    0.833058  


-----EDUCATION----
                               AI21 Labs                              \
Model                          j1-grande  j1-jumbo j1-grande-v2-beta   
EDUCATION                                                              
Less than high school           0.827301  0.828464          0.811735   
High school graduate            0.816720  0.816004          0.799399   
Some college, no degree         0.811134  0.813503          0.803595   
Associate's degree              0.809025  0.810834          0.799860   
College graduate/some postgrad  0.796517  0.801503          0.793541   

                                  OpenAI                         \
Model                                ada   davinci text-ada-001   
EDUCATION                                                         
Less than high school           0.835257  0.800224     0.710170   
High school graduate            0.825634  0.789899     0.710800   
Some college, no degree         0.822550  0.790146     0.705745   
As

-----CITIZEN----
        AI21 Labs                                OpenAI            \
Model   j1-grande  j1-jumbo j1-grande-v2-beta       ada   davinci   
CITIZEN                                                             
Yes      0.811732  0.813693          0.802385  0.822790  0.789382   
No       0.804186  0.816200          0.811977  0.817598  0.797237   

                                                                         \
Model   text-ada-001 text-davinci-001 text-davinci-002 text-davinci-003   
CITIZEN                                                                   
Yes         0.707074         0.713931         0.761843         0.700210   
No          0.699396         0.715292         0.750584         0.706037   

        PersonalLLM  
Model   PersonalLLM  
CITIZEN              
Yes        0.836442  
No         0.832811  


...

-----RELIGATTEND----
                      AI21 Labs                                OpenAI  \
Model                 j1-grande  j1-jumbo j1-grande-v2-beta       ada   
RELIGATTEND                                                             
More than once a week  0.807036  0.806825          0.792659  0.816250   
Once a week            0.808564  0.810543          0.798263  0.818815   
Once or twice a month  0.814401  0.817518          0.807033  0.824936   
A few times a year     0.811946  0.817014          0.808944  0.823552   
Seldom                 0.809034  0.811210          0.800223  0.820661   

                                                               \
Model                   davinci text-ada-001 text-davinci-001   
RELIGATTEND                                                     
More than once a week  0.783669     0.701278         0.708361   
Once a week            0.787299     0.703329         0.711878   
Once or twice a month  0.795281     0.704417         0.713262   
A fe

-----POLPARTY----
            AI21 Labs                                OpenAI            \
Model       j1-grande  j1-jumbo j1-grande-v2-beta       ada   davinci   
POLPARTY                                                                
Republican   0.796694  0.791405          0.776105  0.805351  0.768522   
Democrat     0.792351  0.799684          0.795376  0.803887  0.781268   
Independent  0.809199  0.811751          0.800742  0.821497  0.788248   
Other        0.820406  0.819722          0.803903  0.831813  0.793493   

                                                                             \
Model       text-ada-001 text-davinci-001 text-davinci-002 text-davinci-003   
POLPARTY                                                                      
Republican      0.704608         0.703549         0.741687         0.679510   
Democrat        0.695644         0.714496         0.761735         0.718607   
Independent     0.706380         0.714606         0.763463         0.701344

-----INCOME----
                  AI21 Labs                                OpenAI            \
Model             j1-grande  j1-jumbo j1-grande-v2-beta       ada   davinci   
INCOME                                                                        
Less than $30,000  0.824420  0.827894          0.813289  0.833260  0.800971   
$30,000-$50,000    0.811437  0.814316          0.802455  0.822000  0.789476   
$50,000-$75,000    0.804253  0.806581          0.795529  0.816473  0.783644   
$75,000-$100,000   0.799284  0.800497          0.791074  0.811113  0.780435   
$100,000 or more   0.794315  0.797454          0.789860  0.807461  0.777094   

                                                                  \
Model             text-ada-001 text-davinci-001 text-davinci-002   
INCOME                                                             
Less than $30,000     0.708979         0.716736         0.757221   
$30,000-$50,000       0.707706         0.713243         0.757964   
$50,000-$75

-----POLIDEOLOGY----
                  AI21 Labs                                OpenAI            \
Model             j1-grande  j1-jumbo j1-grande-v2-beta       ada   davinci   
POLIDEOLOGY                                                                   
Very conservative  0.804856  0.796732          0.778405  0.811022  0.771357   
Conservative       0.800078  0.796071          0.780238  0.809650  0.772620   
Moderate           0.809491  0.813633          0.803591  0.822011  0.791382   
Liberal            0.786396  0.791847          0.788254  0.798508  0.773965   
Very liberal       0.780240  0.785363          0.782092  0.791317  0.768062   

                                                                  \
Model             text-ada-001 text-davinci-001 text-davinci-002   
POLIDEOLOGY                                                        
Very conservative     0.701664         0.697720         0.733088   
Conservative          0.706559         0.707161         0.747084   
Modera

-----RACE----
         AI21 Labs                                OpenAI            \
Model    j1-grande  j1-jumbo j1-grande-v2-beta       ada   davinci   
RACE                                                                 
White     0.805646  0.806541          0.794386  0.816783  0.783393   
Black     0.812763  0.820307          0.811583  0.822550  0.795935   
Asian     0.806061  0.813733          0.805680  0.818799  0.792310   
Hispanic  0.812281  0.819917          0.810441  0.824179  0.796970   
Other     0.798470  0.801427          0.782846  0.806710  0.772899   

                                                                          \
Model    text-ada-001 text-davinci-001 text-davinci-002 text-davinci-003   
RACE                                                                       
White        0.707211         0.712410         0.761575         0.698949   
Black        0.700450         0.714338         0.753459         0.702061   
Asian        0.696845         0.714811       

In [26]:
final_table = big_table#.reset_index().reset_index()
final_table

AI21 Labs                              \
Model                          j1-grande  j1-jumbo j1-grande-v2-beta   
Northeast                       0.807440  0.810824          0.801783   
Midwest                         0.808271  0.809645          0.797322   
South                           0.816075  0.818256          0.805385   
West                            0.809897  0.813147          0.802271   
18-29                           0.814438  0.817574          0.807705   
30-49                           0.810853  0.814078          0.804153   
50-64                           0.809033  0.809473          0.796902   
65+                             0.790911  0.792299          0.779172   
Male                            0.812703  0.814410          0.801534   
Female                          0.806857  0.809897          0.799898   
Less than high school           0.827301  0.828464          0.811735   
High school graduate            0.816720  0.816004          0.799399   
Some college, no degree         0.811134  0.813503          0.803595   
Associate's degree              0.809025  0.810834          0.799860   
College graduate/some postgrad  0.796517  0.801503          0.793541   
Postgraduate                    0.787512  0.793655          0.788789   
Yes                             0.811732  0.813693          0.802385   
No                              0.804186  0.816200          0.811977   
Married                         0.807371  0.810225          0.799174   
Divorced                        0.808975  0.809353          0.795940   
Separated                       0.808285  0.814162          0.801379   
Widowed                         0.799103  0.799631          0.784687   
Never been married              0.814908  0.819277          0.807635   
Protestant                      0.810561  0.810216          0.796864   
Roman Catholic                  0.811888  0.816185          0.805620   
Mormon                          0.788448  0.789196          0.776520   
Orthodox                        0.769183  0.773070          0.762268   
Jewish                          0.791207  0.792379          0.784873   
Muslim                          0.783797  0.793598          0.788354   
Buddhist                        0.769907  0.782121          0.776526   
Hindu                           0.776652  0.796376          0.793563   
Atheist                         0.771706  0.774145          0.770718   
Agnostic                        0.781580  0.785107          0.780708   
Other                           0.791445  0.794115          0.789513   
Nothing in particular           0.812942  0.814873          0.801940   
More than once a week           0.807036  0.806825          0.792659   
Once a week                     0.808564  0.810543          0.798263   
Once or twice a month           0.814401  0.817518          0.807033   
A few times a year              0.811946  0.817014          0.808944   
Seldom                          0.809034  0.811210          0.800223   
Never                           0.804359  0.805601          0.794906   
Republican                      0.796694  0.791405          0.776105   
Democrat                        0.792351  0.799684          0.795376   
Independent                     0.809199  0.811751          0.800742   
Other                           0.820406  0.819722          0.803903   
Less than $30,000               0.824420  0.827894          0.813289   
$30,000-$50,000                 0.811437  0.814316          0.802455   
$50,000-$75,000                 0.804253  0.806581          0.795529   
$75,000-$100,000                0.799284  0.800497          0.791074   
$100,000 or more                0.794315  0.797454          0.789860   
Very conservative               0.804856  0.796732          0.778405   
Conservative                    0.800078  0.796071          0.780238   
Moderate                        0.809491  0.813633          0.803591   
Liberal                         0.786396  0.791847          0.788254   

In [27]:
len(final_table)

60

In [28]:
groups_to_keep = [
    'Asian',
    'Black',
    'Hispanic',
    'White',
    'Conservative',
    'Liberal',
    'Democrat',
    'Republican',
    'Muslim',
    'Roman Catholic',
    'Less than $30,000',
    '$100,000 or more',
    '18-29',
    '65+',
    'Less than high school',
    'Postgraduate',
    'Divorced',
    'Married'
]

In [29]:
final_table.loc[groups_to_keep]

AI21 Labs                                OpenAI  \
Model                 j1-grande  j1-jumbo j1-grande-v2-beta       ada   
Asian                  0.806061  0.813733          0.805680  0.818799   
Black                  0.812763  0.820307          0.811583  0.822550   
Hispanic               0.812281  0.819917          0.810441  0.824179   
White                  0.805646  0.806541          0.794386  0.816783   
Conservative           0.800078  0.796071          0.780238  0.809650   
Liberal                0.786396  0.791847          0.788254  0.798508   
Democrat               0.792351  0.799684          0.795376  0.803887   
Republican             0.796694  0.791405          0.776105  0.805351   
Muslim                 0.783797  0.793598          0.788354  0.792245   
Roman Catholic         0.811888  0.816185          0.805620  0.823445   
Less than $30,000      0.824420  0.827894          0.813289  0.833260   
$100,000 or more       0.794315  0.797454          0.789860  0.807461   
18-29                  0.814438  0.817574          0.807705  0.827659   
65+                    0.790911  0.792299          0.779172  0.800263   
Less than high school  0.827301  0.828464          0.811735  0.835257   
Postgraduate           0.787512  0.793655          0.788789  0.800326   
Divorced               0.808975  0.809353          0.795940  0.817044   
Married                0.807371  0.810225          0.799174  0.819468   

                                                               \
Model                   davinci text-ada-001 text-davinci-001   
Asian                  0.792310     0.696845         0.714811   
Black                  0.795935     0.700450         0.714338   
Hispanic               0.796970     0.702914         0.716626   
White                  0.783393     0.707211         0.712410   
Conservative           0.772620     0.706559         0.707161   
Liberal                0.773965     0.695578         0.715702   
Democrat               0.781268     0.695644         0.714496   
Republican             0.768522     0.704608         0.703549   
Muslim                 0.773632     0.681807         0.704655   
Roman Catholic         0.792153     0.706421         0.714301   
Less than $30,000      0.800971     0.708979         0.716736   
$100,000 or more       0.777094     0.698339         0.710093   
18-29                  0.794451     0.704328         0.714203   
65+                    0.770097     0.703954         0.707805   
Less than high school  0.800224     0.710170         0.714818   
Postgraduate           0.774400     0.694581         0.712505   
Divorced               0.784837     0.709067         0.714357   
Married                0.785442     0.705539         0.712234   

                                                        PersonalLLM  
Model                 text-davinci-002 text-davinci-003 PersonalLLM  
Asian                         0.754973         0.707656    0.839145  
Black                         0.753459         0.702061    0.832912  
Hispanic                      0.755039         0.705562    0.838514  
White                         0.761575         0.698949    0.832352  
Conservative                  0.747084         0.683717    0.816992  
Liberal                       0.766625         0.721084    0.833348  
Democrat                      0.761735         0.718607    0.833961  
Republican                    0.741687         0.679510    0.812046  
Muslim                        0.728357         0.697327    0.815607  
Roman Catholic                0.758785         0.701540    0.834798  
Less than $30,000             0.757221         0.692750    0.838481  
$100,000 or more              0.763814         0.708260    0.830994  
18-29                         0.763150         0.700282    0.839955  
65+                           0.751933         0.698711    0.818160  
Less than high school         0.749333         0.685179    0.832223  
Postgraduate                  0.765660         0.716937    0.831377  


In [30]:
print(final_table[[(  'AI21 Labs',          'j1-jumbo'),
            (  'AI21 Labs', 'j1-grande-v2-beta'),
            (     'OpenAI',               'ada'),
            (     'OpenAI',  'text-davinci-003'),
            ('PersonalLLM',       'PersonalLLM')]].to_latex(float_format="%.3f"))

\begin{tabular}{lrrrrr}
\toprule
 & \multicolumn{2}{r}{AI21 Labs} & \multicolumn{2}{r}{OpenAI} & PersonalLLM \\
Model & j1-jumbo & j1-grande-v2-beta & ada & text-davinci-003 & PersonalLLM \\
\midrule
Northeast & 0.811 & 0.802 & 0.819 & 0.704 & 0.838 \\
Midwest & 0.810 & 0.797 & 0.820 & 0.701 & 0.833 \\
South & 0.818 & 0.805 & 0.827 & 0.696 & 0.835 \\
West & 0.813 & 0.802 & 0.821 & 0.704 & 0.839 \\
18-29 & 0.818 & 0.808 & 0.828 & 0.700 & 0.840 \\
30-49 & 0.814 & 0.804 & 0.823 & 0.702 & 0.837 \\
50-64 & 0.809 & 0.797 & 0.818 & 0.696 & 0.830 \\
65+ & 0.792 & 0.779 & 0.800 & 0.699 & 0.818 \\
Male & 0.814 & 0.802 & 0.826 & 0.697 & 0.837 \\
Female & 0.810 & 0.800 & 0.816 & 0.702 & 0.833 \\
Less than high school & 0.828 & 0.812 & 0.835 & 0.685 & 0.832 \\
High school graduate & 0.816 & 0.799 & 0.826 & 0.691 & 0.832 \\
Some college, no degree & 0.814 & 0.804 & 0.823 & 0.701 & 0.836 \\
Associate's degree & 0.811 & 0.800 & 0.821 & 0.700 & 0.834 \\
College graduate/some postgrad & 0.802 & 0.794 & 

In [34]:
print(final_table.loc[groups_to_keep][[(  'AI21 Labs',          'j1-jumbo'),
            (  'AI21 Labs', 'j1-grande-v2-beta'),
            (     'OpenAI',               'ada'),
            (     'OpenAI',  'text-davinci-003'),
            ('PersonalLLM',       'PersonalLLM')]].to_latex(float_format="%.3f"))

\begin{tabular}{lrrrrr}
\toprule
 & \multicolumn{2}{r}{AI21 Labs} & \multicolumn{2}{r}{OpenAI} & PersonalLLM \\
Model & j1-jumbo & j1-grande-v2-beta & ada & text-davinci-003 & PersonalLLM \\
\midrule
Asian & 0.814 & 0.806 & 0.819 & 0.708 & 0.839 \\
Black & 0.820 & 0.812 & 0.823 & 0.702 & 0.833 \\
Hispanic & 0.820 & 0.810 & 0.824 & 0.706 & 0.839 \\
White & 0.807 & 0.794 & 0.817 & 0.699 & 0.832 \\
Conservative & 0.796 & 0.780 & 0.810 & 0.684 & 0.817 \\
Liberal & 0.792 & 0.788 & 0.799 & 0.721 & 0.833 \\
Democrat & 0.800 & 0.795 & 0.804 & 0.719 & 0.834 \\
Republican & 0.791 & 0.776 & 0.805 & 0.680 & 0.812 \\
Muslim & 0.794 & 0.788 & 0.792 & 0.697 & 0.816 \\
Roman Catholic & 0.816 & 0.806 & 0.823 & 0.702 & 0.835 \\
Less than $30,000 & 0.828 & 0.813 & 0.833 & 0.693 & 0.838 \\
$100,000 or more & 0.797 & 0.790 & 0.807 & 0.708 & 0.831 \\
18-29 & 0.818 & 0.808 & 0.828 & 0.700 & 0.840 \\
65+ & 0.792 & 0.779 & 0.800 & 0.699 & 0.818 \\
Less than high school & 0.828 & 0.812 & 0.835 & 0.685 & 0.832 \

In [32]:
final_table[[(  'AI21 Labs',          'j1-jumbo'),
            (  'AI21 Labs', 'j1-grande-v2-beta'),
            (     'OpenAI',               'ada'),
            (     'OpenAI',  'text-davinci-003'),
            ('PersonalLLM',       'PersonalLLM')]]

AI21 Labs                      OpenAI  \
Model                           j1-jumbo j1-grande-v2-beta       ada   
Northeast                       0.810824          0.801783  0.819384   
Midwest                         0.809645          0.797322  0.820396   
South                           0.818256          0.805385  0.826648   
West                            0.813147          0.802271  0.821166   
18-29                           0.817574          0.807705  0.827659   
30-49                           0.814078          0.804153  0.822961   
50-64                           0.809473          0.796902  0.818017   
65+                             0.792299          0.779172  0.800263   
Male                            0.814410          0.801534  0.825710   
Female                          0.809897          0.799898  0.816473   
Less than high school           0.828464          0.811735  0.835257   
High school graduate            0.816004          0.799399  0.825634   
Some college, no degree         0.813503          0.803595  0.822550   
Associate's degree              0.810834          0.799860  0.820599   
College graduate/some postgrad  0.801503          0.793541  0.809647   
Postgraduate                    0.793655          0.788789  0.800326   
Yes                             0.813693          0.802385  0.822790   
No                              0.816200          0.811977  0.817598   
Married                         0.810225          0.799174  0.819468   
Divorced                        0.809353          0.795940  0.817044   
Separated                       0.814162          0.801379  0.817541   
Widowed                         0.799631          0.784687  0.807228   
Never been married              0.819277          0.807635  0.827916   
Protestant                      0.810216          0.796864  0.820130   
Roman Catholic                  0.816185          0.805620  0.823445   
Mormon                          0.789196          0.776520  0.801639   
Orthodox                        0.773070          0.762268  0.781282   
Jewish                          0.792379          0.784873  0.800023   
Muslim                          0.793598          0.788354  0.792245   
Buddhist                        0.782121          0.776526  0.782797   
Hindu                           0.796376          0.793563  0.789134   
Atheist                         0.774145          0.770718  0.784293   
Agnostic                        0.785107          0.780708  0.794005   
Other                           0.794115          0.789513  0.800505   
Nothing in particular           0.814873          0.801940  0.824435   
More than once a week           0.806825          0.792659  0.816250   
Once a week                     0.810543          0.798263  0.818815   
Once or twice a month           0.817518          0.807033  0.824936   
A few times a year              0.817014          0.808944  0.823552   
Seldom                          0.811210          0.800223  0.820661   
Never                           0.805601          0.794906  0.815547   
Republican                      0.791405          0.776105  0.805351   
Democrat                        0.799684          0.795376  0.803887   
Independent                     0.811751          0.800742  0.821497   
Other                           0.819722          0.803903  0.831813   
Less than $30,000               0.827894          0.813289  0.833260   
$30,000-$50,000                 0.814316          0.802455  0.822000   
$50,000-$75,000                 0.806581          0.795529  0.816473   
$75,000-$100,000                0.800497          0.791074  0.811113   
$100,000 or more                0.797454          0.789860  0.807461   
Very conservative               0.796732          0.778405  0.811022   
Conservative                    0.796071          0.780238  0.809650   
Moderate                        0.813633          0.803591  0.822011   
Liberal                         0.791847          0.788254  0.798508   

In [33]:
[(  'AI21 Labs',          'j1-jumbo'),
            (  'AI21 Labs', 'j1-grande-v2-beta'),
            (     'OpenAI',               'ada'),
            (     'OpenAI',  'text-davinci-003'),
            ('PersonalLLM',       'PersonalLLM')]

[('AI21 Labs', 'j1-jumbo'),
 ('AI21 Labs', 'j1-grande-v2-beta'),
 ('OpenAI', 'ada'),
 ('OpenAI', 'text-davinci-003'),
 ('PersonalLLM', 'PersonalLLM')]

In [53]:
ctr = 0

for row in final_table.reset_index().iterrows():
    print()
    vals = list(row[1])[1:]
    if np.max(vals) == vals[-1]:
        ctr += 1

ctr

59